# Polynomial Regression

A straight line cannot follow a curve. Polynomial regression adds columns for x², x³ and so on, then runs the same linear regression on all of them. Here it predicts **mpg** from **horsepower**, which curves.

This notebook goes with the **Polynomial Regression** lesson on [ML Visual](https://mohammad-qt.github.io/ML_DL_Vis/). Run each cell in order with **Shift + Enter**. Nothing to install: Colab already has every library used here.

**The steps:** load the data → split → add the power columns → fit → compare degrees → draw the curve.

## 1. Load the data

Each row is a car from the late 1970s and early 1980s. We predict `mpg` from `horsepower`. A few cars have no horsepower listed, so we drop them first.

The data is the [Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg) data set from the UCI Machine Learning Repository (CC BY 4.0). See `datasets/README.md` for what we changed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# The data lives in this project's GitHub repository, so there is nothing to upload.
DATA_URL = "https://raw.githubusercontent.com/Mohammad-QT/ML_DL_Vis/main/datasets/mpg.csv"
dataset = pd.read_csv(DATA_URL)
dataset.head()

In [ ]:
dataset = dataset.dropna(subset=['horsepower'])
X = dataset[['horsepower']].values
y = dataset['mpg'].values
print(X.shape, 'cars')

## 2. Split into train and test

The model learns from the **training** cars (75%). We hold back the other 25% and only use them at the end. The test score shows how the model does on cars it has never seen. These are the ringed points on the site's boards.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
print(len(X_train), 'train,', len(X_test), 'test')

## 3. Scale, then add the power columns and fit

Horsepower is in the hundreds, so its 8th power is astronomically large and the fit becomes unstable. We **scale x first** (fit on the training cars only), then build the powers.

`PolynomialFeatures(degree=2)` turns each x into the columns 1, x, x². The regression on top of it is still linear in the weights.

In [ ]:
sc = StandardScaler()
X_train_s = sc.fit_transform(X_train)
X_test_s = sc.transform(X_test)

poly = PolynomialFeatures(degree=2)
X_poly_train = poly.fit_transform(X_train_s)
regressor = LinearRegression()
regressor.fit(X_poly_train, y_train)
print('columns:', poly.get_feature_names_out())
print('weights:', regressor.coef_.round(4))

## 4. Check the result

**R²** says how much of the ups and downs in the data the model explains. 1 is perfect, 0 is no better than always guessing the average. The number that matters is the **test** R². The last line shows the typical miss in miles per gallon.

In [ ]:
y_pred_train = regressor.predict(poly.transform(X_train_s))
y_pred_test = regressor.predict(poly.transform(X_test_s))
print('Train R²:', round(r2_score(y_train, y_pred_train), 3))
print('Test R²: ', round(r2_score(y_test, y_pred_test), 3))
print('Test error on average:', round(mean_absolute_error(y_test, y_pred_test), 2), 'mpg')

## 5. Choose the degree

Too low and the curve misses the pattern. Too high and it bends to follow noise. Compare train and test R² for each degree. Raise the degree only while the **test** score keeps rising.

In [ ]:
for degree in range(1, 9):
    p = PolynomialFeatures(degree=degree)
    m = LinearRegression().fit(p.fit_transform(X_train_s), y_train)
    tr = r2_score(y_train, m.predict(p.transform(X_train_s)))
    te = r2_score(y_test, m.predict(p.transform(X_test_s)))
    print(f'degree {degree}: train R² {tr:.3f}   test R² {te:.3f}')

In [ ]:
grid = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
plt.figure(figsize=(7, 4.5))
plt.scatter(X_train, y_train, color='#5b7cfa', label='train')
plt.scatter(X_test, y_test, facecolors='white', edgecolors='#5b7cfa', linewidths=1.8, label='test')
for degree, style in [(1, '--'), (2, '-'), (8, ':')]:
    p = PolynomialFeatures(degree=degree)
    m = LinearRegression().fit(p.fit_transform(X_train_s), y_train)
    plt.plot(grid, m.predict(p.transform(sc.transform(grid))), style, color='black', label=f'degree {degree}')
plt.xlabel('horsepower'); plt.ylabel('mpg'); plt.legend()
plt.show()

## Try this

1. Which degree has the best test R²? Does it change if you change `random_state`?
2. Look at the degree 8 curve at the right-hand end, where there are few cars. What does it do?
3. Predict the mpg of a 130 hp car with your best degree.